<a href="https://colab.research.google.com/github/osergioribeirof/Python/blob/main/SR_GammaFlip_Interativo_BARCHART.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### BIBLIOTECA

In [1]:
### CÓDIGO ADAPTADO PARA BARCHART - ESTRUTURA ORIGINAL COMPLETA ###

# ========== CÉLULA 1: INSTALAR PLOTLY ==========
### Rodar essa célula somente uma vez ###
#!pip install plotly

# ========================================
# CÉLULA 3 - IMPORTS
# ========================================
import pandas as pd
import plotly
pd.set_option('plotting.backend','plotly')
import plotly.graph_objs as go
import numpy as np
import scipy
from scipy.stats import norm
#import matplotlib.pyplot as plt
import calendar
from datetime import datetime, timedelta, date

### Arquivo CSV

In [2]:
# ========================================
# CÉLULA 4 - FORMATO DISPLAY
# ========================================
pd.options.display.float_format = '{:,.4f}'.format

In [3]:
# ========================================
# CÉLULA 6 - FUNÇÕES E PARÂMETROS
# ========================================
# Parametros de entrada
filename = 'nqz25-volatility-greeks-exp-12_19_25-50-strikes-_-10-29-2025.csv'

In [4]:
# Black-Scholes European-Options Gamma
def calcGammaEx(S, K, vol, T, r, q, optType, OI):
    if T == 0 or vol == 0:
        return 0

    dp = (np.log(S/K) + (r - q + 0.5*vol**2)*T) / (vol*np.sqrt(T))
    dm = dp - vol*np.sqrt(T)

    if optType == 'call':
        gamma = np.exp(-q*T) * norm.pdf(dp) / (S * vol * np.sqrt(T))
        return OI * 100 * S * S * 0.01 * gamma
    else:
        gamma = K * np.exp(-r*T) * norm.pdf(dm) / (S * S * vol * np.sqrt(T))
        return OI * 100 * S * S * 0.01 * gamma

def isThirdFriday(d):
    return d.weekday() == 4 and 15 <= d.day <= 21

In [6]:
# ========================================
# CÉLULA 7 - MARKDOWN
# ========================================
# ### PREPARAÇÃO DO ARQUIVO

# ========================================
# CÉLULA 8 - PROCESSAR CSV BARCHART
# ========================================
# ADAPTAÇÃO: Ler CSV do Barchart
df_raw = pd.read_csv('/content/nqz25-volatility-greeks-exp-12_19_25-50-strikes-+_--10-29-2025.csv')
df_raw = df_raw[df_raw['Type'].notna()].copy()

# DEFINA O SPOT PRICE MANUALMENTE (Barchart não inclui no CSV)
spotPrice = 26200.00  # ⚠️ AJUSTE AQUI O PREÇO ATUAL DO NQ

fromStrike = 0.8 * spotPrice
toStrike = 1.2 * spotPrice

# Data de hoje
todayDate = datetime.now()

# Limpar colunas do Barchart
df_raw['Strike'] = df_raw['Strike'].str.replace(',', '').astype(float)
df_raw['IV'] = df_raw['IV'].str.replace('%', '').astype(float) / 100

# ⚠️ IMPORTANTE: Adicionar Open Interest (Barchart não tem)
df_raw['OpenInt'] = 100  # AJUSTE AQUI se tiver OI de outra fonte

# Separar calls e puts
df_calls = df_raw[df_raw['Type'] == 'Call'].copy()
df_puts = df_raw[df_raw['Type'] == 'Put'].copy()

# Data de expiração (ajuste conforme seu CSV)
expiration_date = datetime(2025, 12, 19, 16, 0)  # ⚠️ AJUSTE A DATA DE VENCIMENTO

# Criar DataFrame no formato original (linha por strike com call e put)
strikes_unique = sorted(df_raw['Strike'].unique())
data_list = []

for strike in strikes_unique:
    call_row = df_calls[df_calls['Strike'] == strike]
    put_row = df_puts[df_puts['Strike'] == strike]

    row_data = {
        'ExpirationDate': expiration_date,
        'StrikePrice': strike,
        'CallIV': call_row['IV'].values[0] if len(call_row) > 0 else 0,
        'PutIV': put_row['IV'].values[0] if len(put_row) > 0 else 0,
        'CallGamma': call_row['Gamma'].values[0] if len(call_row) > 0 else 0,
        'PutGamma': put_row['Gamma'].values[0] if len(put_row) > 0 else 0,
        'CallOpenInt': call_row['OpenInt'].values[0] if len(call_row) > 0 else 0,
        'PutOpenInt': put_row['OpenInt'].values[0] if len(put_row) > 0 else 0,
        'CallDelta': call_row['Delta'].values[0] if len(call_row) > 0 else 0,
        'PutDelta': put_row['Delta'].values[0] if len(put_row) > 0 else 0,
    }
    data_list.append(row_data)

df = pd.DataFrame(data_list)

# Calcular dias até expiração (em dias úteis / 262)
df['daysTillExp'] = df['ExpirationDate'].apply(
    lambda x: np.busday_count(todayDate.date(), x.date()) / 262
)

### GAMMA - GEX

In [7]:
# ========================================
# CÉLULA 9 - MARKDOWN
# ========================================
# ### GAMMA - GEX

# ========================================
# CÉLULA 10 - CALCULAR GEX
# ========================================
# ---=== CALCULATE SPOT GAMMA ===---
df['CallGEX'] = df['CallGamma'] * df['CallOpenInt'] * 100 * spotPrice * spotPrice * 0.01
df['PutGEX'] = df['PutGamma'] * df['PutOpenInt'] * 100 * spotPrice * spotPrice * 0.01 * -1

df['TotalGamma'] = (df.CallGEX + df.PutGEX) / 10**9
dfAgg = df.groupby(['StrikePrice']).sum(numeric_only=True)
strikes = dfAgg.index.values

In [8]:
# ========================================
# CÉLULA 11 - GRÁFICO 1: ABSOLUTE GAMMA EXPOSURE
# ========================================
# Chart 1: Absolute Gamma Exposure
x_data = strikes
y_data = dfAgg['TotalGamma'].to_numpy()

fig = go.Figure(
    go.Bar(
        x=x_data,
        y=y_data,
        width=6,
        marker_color='rgb(26, 118, 255)',
        marker_line_color='black',
        marker_line_width=0.15,
        name='Gamma Exposure'
    )
)

fig.add_shape(
    type='line',
    x0=spotPrice,
    y0=min(y_data),
    x1=spotPrice,
    y1=max(y_data),
    line=dict(color='red', width=2, dash='dash')
)

fig.update_layout(
    title={
        'text': f"Total Gamma: ${df['TotalGamma'].sum():,.2f} Bn per 1% Ativo Move",
        'font': {'size': 20, 'family': 'Arial Black',}
    },
    xaxis_title='Strike',
    yaxis_title='Spot Gamma Exposure ($ billions/1% move)',
    xaxis=dict(range=[fromStrike, toStrike]),
    yaxis=dict(tickformat='$,.2f'),
    plot_bgcolor='white',
    font=dict(family='Arial', size=12, color='black')
)

fig.update_layout(width=1750, height=800)
fig.show()

In [9]:
# ========================================
# CÉLULA 12 - ANÁLISE GRÁFICO 1
# ========================================
# DADOS DO CHART 1
dfAgg_sorted = dfAgg.sort_values(by='TotalGamma')
# Get the 3 smallest gamma values
smallest_gamma = dfAgg_sorted.head(3)
print("Menores valores de Gamma:")
print(f"  Put Wall: {smallest_gamma.iloc[0]['TotalGamma']:.4f} at strike {smallest_gamma.index[0]:.2f}")
print(f"  Large Gamma: {smallest_gamma.iloc[1]['TotalGamma']:.4f} at strike {smallest_gamma.index[1]:.2f}")
print(f"  Large Gamma: {smallest_gamma.iloc[2]['TotalGamma']:.4f} at strike {smallest_gamma.index[2]:.2f}")

# Get the 3 largest gamma values
largest_gamma = dfAgg_sorted.tail(3)
print("\nMaiores valores de Gamma:")
print(f"  Call Wall: {largest_gamma.iloc[2]['TotalGamma']:.4f} at strike {largest_gamma.index[2]:.2f}")
print(f"  Large Gamma: {largest_gamma.iloc[1]['TotalGamma']:.4f} at strike {largest_gamma.index[1]:.2f}")
print(f"  Large Gamma: {largest_gamma.iloc[0]['TotalGamma']:.4f} at strike {largest_gamma.index[0]:.2f}")


Menores valores de Gamma:
  Put Wall: -0.0038 at strike 23300.00
  Large Gamma: -0.0037 at strike 23200.00
  Large Gamma: -0.0037 at strike 23250.00

Maiores valores de Gamma:
  Call Wall: 0.0056 at strike 25900.00
  Large Gamma: 0.0036 at strike 26400.00
  Large Gamma: 0.0036 at strike 26100.00


In [10]:
# ========================================
# CÉLULA 13 - GRÁFICO 2: GAMMA BY CALLS AND PUTS
# ========================================
# Chart 2: Absolute Gamma Exposure by Calls and Puts
fig = go.Figure()
fig.add_bar(x=strikes, y=dfAgg['CallGEX'].to_numpy() / 10**9, width=6, name="Call Gamma")
fig.add_bar(x=strikes, y=dfAgg['PutGEX'].to_numpy() / 10**9, width=6, name="Put Gamma")
fig.update_xaxes(range=[fromStrike, toStrike])

chartTitle = "Total Gamma: $" + str("{:.2f}".format(df['TotalGamma'].sum())) + " Bn per 1% SPX Move"
fig.update_layout(title_text=chartTitle, title_font=dict(size=20, family="Arial Black"))
fig.update_xaxes(title_text="Strike")
fig.update_yaxes(title_text="Spot Gamma Exposure ($ billions/1% move)")

fig.add_shape(dict(
    type="line",
    x0=spotPrice,
    y0=0,
    x1=spotPrice,
    y1=max(dfAgg['CallGEX'].to_numpy() / 10**9),
    line=dict(color="black", width=2),
    name="SPX Spot:" + str("{:,.0f}".format(spotPrice))
))

fig.update_layout(width=1750, height=800)
fig.show()


In [11]:
# ========================================
# CÉLULA 14 - ANÁLISE GRÁFICO 2
# ========================================
# DADOS CHART 2
dfAgg['AbsoluteTotalGEX'] = dfAgg['CallGEX'].abs() + dfAgg['PutGEX'].abs()
# Sort by AbsoluteTotalGEX to find the strikes with the largest combined exposure
dfAgg_sorted_gex = dfAgg.sort_values(by='AbsoluteTotalGEX', ascending=False)
# Get the top 6 strikes based on combined absolute GEX
gex_levels = dfAgg_sorted_gex.head(6)
print("Top 6 GEX Levels (based on sum of absolute Call and Put Gamma Exposure):")
for idx, strike in enumerate(gex_levels.index, 1):
    print(f"  {idx}. Strike {strike:.2f}: Call GEX = {gex_levels.loc[strike, 'CallGEX']/10**9:.4f}, Put GEX = {gex_levels.loc[strike, 'PutGEX']/10**9:.4f}, Total Absolute = {gex_levels.loc[strike, 'AbsoluteTotalGEX']/10**9:.4f}")

Top 6 GEX Levels (based on sum of absolute Call and Put Gamma Exposure):
  1. Strike 26300.00: Call GEX = 0.0171, Put GEX = -0.0141, Total Absolute = 0.0312
  2. Strike 25900.00: Call GEX = 0.0182, Put GEX = -0.0126, Total Absolute = 0.0307
  3. Strike 26200.00: Call GEX = 0.0166, Put GEX = -0.0135, Total Absolute = 0.0301
  4. Strike 26400.00: Call GEX = 0.0168, Put GEX = -0.0132, Total Absolute = 0.0300
  5. Strike 26100.00: Call GEX = 0.0168, Put GEX = -0.0132, Total Absolute = 0.0300
  6. Strike 26000.00: Call GEX = 0.0154, Put GEX = -0.0136, Total Absolute = 0.0290


In [12]:
# ========================================
# CÉLULA 15 - CALCULAR GAMMA PROFILE
# ========================================
# For each spot level, calculate gamma exposure by applying gamma exposure at that strike
levels = np.linspace(fromStrike, toStrike, 60)

# Expiração próxima e mensal
nextExpiry = df['ExpirationDate'].min()
df["isThirdFriday"] = df['ExpirationDate'].apply(isThirdFriday)
thirdFridays = df.loc[df["isThirdFriday"] == True]
nextMonthlyExp = thirdFridays['ExpirationDate'].min() if len(thirdFridays) > 0 else nextExpiry

totalGamma = []
totalGammaExNext = []
totalGammaExFri = []

# Para cada nível de preço
for level in levels:
    df['callGammaEx'] = df.apply(lambda row: calcGammaEx(
        level, row['StrikePrice'], row['CallIV'],
        (row['ExpirationDate'] - todayDate).days / 365,
        0, 0, 'call', row['CallOpenInt']
    ), axis=1)

    df['putGammaEx'] = df.apply(lambda row: calcGammaEx(
        level, row['StrikePrice'], row['PutIV'],
        (row['ExpirationDate'] - todayDate).days / 365,
        0, 0, 'put', row['PutOpenInt']
    ), axis=1)

    totalGamma.append((df['callGammaEx'].sum() - df['putGammaEx'].sum()) / 10**9)

    # Ex-Next Expiry
    totalGammaExNext.append(
        (df.loc[df['ExpirationDate'] != nextExpiry, 'callGammaEx'].sum() -
         df.loc[df['ExpirationDate'] != nextExpiry, 'putGammaEx'].sum()) / 10**9
    )

    # Ex-Next Monthly
    totalGammaExFri.append(
        (df.loc[df['ExpirationDate'] != nextMonthlyExp, 'callGammaEx'].sum() -
         df.loc[df['ExpirationDate'] != nextMonthlyExp, 'putGammaEx'].sum()) / 10**9
    )


In [13]:
# ========================================
# CÉLULA 16 - GRÁFICO 3: GAMMA EXPOSURE PROFILE
# ========================================
# Chart 3: Gamma Exposure Profile
fig = go.Figure()

fig.add_trace(go.Scatter(x=levels, y=totalGamma, mode='lines', name='All Expiries'))
fig.add_trace(go.Scatter(x=levels, y=totalGammaExNext, mode='lines', name='Ex-Next Expiry'))
fig.add_trace(go.Scatter(x=levels, y=totalGammaExFri, mode='lines', name='Ex-Next Monthly Expiry'))

chartTitle = "Gamma Exposure Profile, SPX, " + todayDate.strftime('%d %b %Y')
fig.update_layout(
    title=chartTitle,
    xaxis_title='Index Price',
    yaxis_title='Gamma Exposure ($ billions/1% move)',
    title_font=dict(size=20, family="Arial Black")
)

fig.add_shape(dict(
    type="line",
    x0=spotPrice,
    y0=min(totalGamma),
    x1=spotPrice,
    y1=max(totalGamma),
    line=dict(color="red", width=2, dash="dash")
))

fig.update_layout(width=1750, height=800, plot_bgcolor='white')
fig.show()


In [14]:
# ========================================
# CÉLULA 17 - ANÁLISE GRÁFICO 3
# ========================================
# DADOS CHART 3
# Find the point on the 'Ex-Next Monthly Expiry' line closest to the spot price (green line and red dotted line intersection as defined by user)
closest_level_index_green_red = np.abs(levels - spotPrice).argmin()
gamma_at_spot_green_red = totalGammaExFri[closest_level_index_green_red]
gamma_at_spot_strike_green_red = levels[closest_level_index_green_red]

print(f"Gamma Flip (Ex-Next Monthly Expiry): {gamma_at_spot_green_red:.4f} at strike {gamma_at_spot_strike_green_red:.2f} (Intersection of Green and Red lines)")

Gamma Flip (Ex-Next Monthly Expiry): 0.0000 at strike 26111.19 (Intersection of Green and Red lines)


In [15]:
# ========================================
# CÉLULA 18 - CONSOLIDAÇÃO RESULTADOS GAMMA
# ========================================
# Consolidating results from CHART 1, CHART 2, and CHART 3
print("--- DADOS CHART 1 ---")
# DADOS CHART 1
dfAgg_sorted = dfAgg.sort_values(by='TotalGamma')
smallest_gamma = dfAgg_sorted.head(3)
print("Menores valores de Gamma:")
print(f"  Put Wall: {smallest_gamma.iloc[0]['TotalGamma']:.4f} at strike {smallest_gamma.index[0]:.2f}")
print(f"  Large Gamma: {smallest_gamma.iloc[1]['TotalGamma']:.4f} at strike {smallest_gamma.index[1]:.2f}")
print(f"  Large Gamma: {smallest_gamma.iloc[2]['TotalGamma']:.4f} at strike {smallest_gamma.index[2]:.2f}")

largest_gamma = dfAgg_sorted.tail(3)
print("\nMaiores valores de Gamma:")
print(f"  Call Wall: {largest_gamma.iloc[2]['TotalGamma']:.4f} at strike {largest_gamma.index[2]:.2f}")
print(f"  Large Gamma: {largest_gamma.iloc[1]['TotalGamma']:.4f} at strike {largest_gamma.index[1]:.2f}")
print(f"  Large Gamma: {largest_gamma.iloc[0]['TotalGamma']:.4f} at strike {largest_gamma.index[0]:.2f}")

print("\n--- DADOS CHART 2 ---")
print("Top 6 GEX Levels:")
for idx, strike in enumerate(gex_levels.index, 1):
    print(f"  {idx}. Strike {strike:.2f}: Total Absolute = {gex_levels.loc[strike, 'AbsoluteTotalGEX']/10**9:.4f}")

print("\n--- DADOS CHART 3 ---")
# Include Gamma Flip from CÉLULA 17
print(f"Gamma Flip (Ex-Next Monthly Expiry): {gamma_at_spot_green_red:.4f} at strike {gamma_at_spot_strike_green_red:.2f}")

--- DADOS CHART 1 ---
Menores valores de Gamma:
  Put Wall: -0.0038 at strike 23300.00
  Large Gamma: -0.0037 at strike 23200.00
  Large Gamma: -0.0037 at strike 23250.00

Maiores valores de Gamma:
  Call Wall: 0.0056 at strike 25900.00
  Large Gamma: 0.0036 at strike 26400.00
  Large Gamma: 0.0036 at strike 26100.00

--- DADOS CHART 2 ---
Top 6 GEX Levels:
  1. Strike 26300.00: Total Absolute = 0.0312
  2. Strike 25900.00: Total Absolute = 0.0307
  3. Strike 26200.00: Total Absolute = 0.0301
  4. Strike 26400.00: Total Absolute = 0.0300
  5. Strike 26100.00: Total Absolute = 0.0300
  6. Strike 26000.00: Total Absolute = 0.0290

--- DADOS CHART 3 ---
Gamma Flip (Ex-Next Monthly Expiry): 0.0000 at strike 26111.19


### DELTA - DEX

In [19]:
# ========================================
# CÉLULA 21 - MARKDOWN
# ========================================
# ### DELTA - DEX

# ========================================
# CÉLULA 22 - CALCULAR DEX
# ========================================
# ---=== CALCULATE SPOT DELTA ===---
df['CallDEX'] = df['CallDelta'] * df['CallOpenInt'] * 100 * spotPrice * 0.01
df['PutDEX'] = df['PutDelta'] * df['PutOpenInt'] * 100 * spotPrice * 0.01

df['TotalDelta'] = (df.CallDEX + df.PutDEX) / 10**6
dfAgg_delta = df.groupby(['StrikePrice']).sum(numeric_only=True)
strikes_delta = dfAgg_delta.index.values

In [20]:
# ========================================
# CÉLULA 23 - GRÁFICO 4: ABSOLUTE DELTA EXPOSURE
# ========================================
# Chart 4: Absolute Delta Exposure
x_data_delta = strikes_delta
y_data_delta = dfAgg_delta['TotalDelta'].to_numpy()

fig_delta4 = go.Figure(
    go.Bar(
        x=x_data_delta,
        y=y_data_delta,
        width=6,
        marker_color='rgb(26, 118, 255)',
        marker_line_color='black',
        marker_line_width=0.15,
        name='Delta Exposure'
    )
)

fig_delta4.add_shape(
    type='line',
    x0=spotPrice,
    y0=min(y_data_delta),
    x1=spotPrice,
    y1=max(y_data_delta),
    line=dict(color='red', width=2, dash='dash')
)

fig_delta4.update_layout(
    title={
        'text': f"Total Delta: ${df['TotalDelta'].sum():,.2f} Million per 1% Ativo Move",
        'font': {'size': 20, 'family': 'Arial Black'}
    },
    xaxis_title='Strike',
    yaxis_title='Spot Delta Exposure ($ millions/1% move)',
    xaxis=dict(range=[fromStrike, toStrike]),
    yaxis=dict(tickformat='$,.2f'),
    plot_bgcolor='white',
    font=dict(family='Arial', size=12, color='black')
)

fig_delta4.update_layout(width=1750, height=800)
fig_delta4.show()

In [21]:
# ========================================
# CÉLULA 24 - GRÁFICO 5: DELTA BY CALLS AND PUTS
# ========================================
# Chart 5: Absolute Delta Exposure by Calls and Puts
fig_delta5 = go.Figure()
fig_delta5.add_bar(x=strikes_delta, y=dfAgg_delta['CallDEX'].to_numpy() / 10**6, width=6, name="Call Delta")
fig_delta5.add_bar(x=strikes_delta, y=dfAgg_delta['PutDEX'].to_numpy() / 10**6, width=6, name="Put Delta")
fig_delta5.update_xaxes(range=[fromStrike, toStrike])

chartTitle_delta5 = "Total Delta: $" + str("{:.2f}".format(df['TotalDelta'].sum())) + " Million per 1% SPX Move"
fig_delta5.update_layout(title_text=chartTitle_delta5, title_font=dict(size=20, family="Arial Black"))
fig_delta5.update_xaxes(title_text="Strike")
fig_delta5.update_yaxes(title_text="Spot Delta Exposure ($ millions/1% move)")

fig_delta5.add_shape(dict(
    type="line",
    x0=spotPrice,
    y0=min(dfAgg_delta['PutDEX'].to_numpy() / 10**6),
    x1=spotPrice,
    y1=max(dfAgg_delta['CallDEX'].to_numpy() / 10**6),
    line=dict(color="black", width=2)
))

fig_delta5.update_layout(width=1750, height=800)
fig_delta5.show()

In [22]:
# ========================================
# CÉLULA 25 - CALCULAR DELTA PROFILE
# ========================================
levels_delta = np.linspace(fromStrike, toStrike, 60)

totalDelta = []
totalDeltaExNext = []
totalDeltaExFri = []

for level in levels_delta:
    # Para Delta, usamos o delta já calculado (simplificação)
    totalDelta.append(df['TotalDelta'].sum())
    totalDeltaExNext.append(
        df.loc[df['ExpirationDate'] != nextExpiry, 'TotalDelta'].sum()
    )
    totalDeltaExFri.append(
        df.loc[df['ExpirationDate'] != nextMonthlyExp, 'TotalDelta'].sum()
    )


In [23]:
# ========================================
# CÉLULA 26 - GRÁFICO 6: DELTA EXPOSURE PROFILE
# ========================================
# Chart 6: Delta Exposure Profile
fig_delta6 = go.Figure()

fig_delta6.add_trace(go.Scatter(x=levels_delta, y=totalDelta, mode='lines', name='All Expiries'))
fig_delta6.add_trace(go.Scatter(x=levels_delta, y=totalDeltaExNext, mode='lines', name='Ex-Next Expiry'))
fig_delta6.add_trace(go.Scatter(x=levels_delta, y=totalDeltaExFri, mode='lines', name='Ex-Next Monthly Expiry'))

chartTitle_delta6 = "Delta Exposure Profile, SPX, " + todayDate.strftime('%d %b %Y')
fig_delta6.update_layout(
    title=chartTitle_delta6,
    xaxis_title='Index Price',
    yaxis_title='Delta Exposure ($ millions/1% move)',
    title_font=dict(size=20, family="Arial Black")
)

fig_delta6.add_shape(dict(
    type="line",
    x0=spotPrice,
    y0=min(totalDelta),
    x1=spotPrice,
    y1=max(totalDelta),
    line=dict(color="red", width=2, dash="dash")
))

fig_delta6.update_layout(width=1750, height=800, plot_bgcolor='white')
fig_delta6.show()


In [36]:
# ========================================
# CÉLULA NOVA - ANÁLISE DELTA FLIP
# ========================================
# Find the point on the 'Ex-Next Monthly Expiry' line closest to the spot price
closest_level_index_delta_fri = np.abs(levels_delta - spotPrice).argmin()
delta_at_spot_delta_fri = totalDeltaExFri[closest_level_index_delta_fri]
delta_at_spot_strike_delta_fri = levels_delta[closest_level_index_delta_fri]

# Find the point on the 'All Expiries' line closest to the spot price
closest_level_index_delta_all = np.abs(levels_delta - spotPrice).argmin()
delta_at_spot_delta_all = totalDelta[closest_level_index_delta_all]
delta_at_spot_strike_delta_all = levels_delta[closest_level_index_delta_all]


print("--- DELTA FLIP POINTS ---")
print(f"Delta Flip (Ex-Next Monthly Expiry): {delta_at_spot_delta_fri:.4f} at strike {delta_at_spot_strike_delta_fri:.2f}")
print(f"Delta Flip (All Expiries): {delta_at_spot_delta_all:.4f} at strike {delta_at_spot_strike_delta_all:.2f}")

--- DELTA FLIP POINTS ---
Delta Flip (Ex-Next Monthly Expiry): 0.0000 at strike 26111.19
Delta Flip (All Expiries): 34.3712 at strike 26111.19


In [24]:
# ========================================
# CÉLULA 27 - ANÁLISE DELTA
# ========================================
# Consolidating results from CHART 4, CHART 5, and CHART 6
print("--- DADOS CHART 4 ---")
# Find the 3 strikes with the smallest (most negative) total Delta Exposure
smallest_delta = dfAgg_delta.sort_values(by='TotalDelta').head(3)
print("Menores valores de Delta:")
print(f"  Put Delta Wall: {smallest_delta.iloc[0]['TotalDelta']:.4f} at strike {smallest_delta.index[0]:.2f}")
print(f"  Large Delta: {smallest_delta.iloc[1]['TotalDelta']:.4f} at strike {smallest_delta.index[1]:.2f}")
print(f"  Large Delta: {smallest_delta.iloc[2]['TotalDelta']:.4f} at strike {smallest_delta.index[2]:.2f}")

# Find the 3 strikes with the largest (most positive) total Delta Exposure
largest_delta = dfAgg_delta.sort_values(by='TotalDelta').tail(3)
print("\nMaiores valores de Delta:")
print(f"  Call Delta Wall: {largest_delta.iloc[2]['TotalDelta']:.4f} at strike {largest_delta.index[2]:.2f}")
print(f"  Large Delta: {largest_delta.iloc[1]['TotalDelta']:.4f} at strike {largest_delta.index[1]:.2f}")
print(f"  Large Delta: {largest_delta.iloc[0]['TotalDelta']:.4f} at strike {largest_delta.index[0]:.2f}")


--- DADOS CHART 4 ---
Menores valores de Delta:
  Put Delta Wall: -2.3695 at strike 35500.00
  Large Delta: -2.3644 at strike 35000.00
  Large Delta: -2.3586 at strike 34500.00

Maiores valores de Delta:
  Call Delta Wall: 2.4483 at strike 22250.00
  Large Delta: 2.4436 at strike 22200.00
  Large Delta: 2.4354 at strike 22300.00


### RESULTADOS GAMMA

In [35]:
# ========================================
# GERADOR TRADINGVIEW - GAMMA (ALL) - VERSÃO COMPLETA
# ========================================
print("\n" + "="*80)
print("🚀 GERADOR DE CÓDIGO TRADINGVIEW - GAMMA FLIP (VERSÃO SIMPLIFICADA)")
print("="*80 + "\n")

# ==================== COLETA DOS DADOS ====================

# CHART 1 - Spot Gamma Levels (6 strikes: 3 menores + 3 maiores)
c1_put_wall_value = smallest_gamma.iloc[0]['TotalGamma']
c1_put_wall_strike = smallest_gamma.index[0]
c1_lg1_value = smallest_gamma.iloc[1]['TotalGamma']
c1_lg1_strike = smallest_gamma.index[1]
c1_lg2_value = smallest_gamma.iloc[2]['TotalGamma']
c1_lg2_strike = smallest_gamma.index[2]

c1_call_wall_value = largest_gamma.iloc[2]['TotalGamma']
c1_call_wall_strike = largest_gamma.index[2]
c1_lg3_value = largest_gamma.iloc[1]['TotalGamma']
c1_lg3_strike = largest_gamma.index[1]
c1_lg4_value = largest_gamma.iloc[0]['TotalGamma']
c1_lg4_strike = largest_gamma.index[0]

# CHART 2 - GEX Levels (Top 6 strikes por GEX absoluto)
c2_gex1_strike = gex_levels.index[0]
c2_gex1_call = gex_levels.iloc[0]['CallGEX'] / 10**9
c2_gex1_put = gex_levels.iloc[0]['PutGEX'] / 10**9

c2_gex2_strike = gex_levels.index[1]
c2_gex2_call = gex_levels.iloc[1]['CallGEX'] / 10**9
c2_gex2_put = gex_levels.iloc[1]['PutGEX'] / 10**9

c2_gex3_strike = gex_levels.index[2]
c2_gex3_call = gex_levels.iloc[2]['CallGEX'] / 10**9
c2_gex3_put = gex_levels.iloc[2]['PutGEX'] / 10**9

c2_gex4_strike = gex_levels.index[3]
c2_gex4_call = gex_levels.iloc[3]['CallGEX'] / 10**9
c2_gex4_put = gex_levels.iloc[3]['PutGEX'] / 10**9

c2_gex5_strike = gex_levels.index[4]
c2_gex5_call = gex_levels.iloc[4]['CallGEX'] / 10**9
c2_gex5_put = gex_levels.iloc[4]['PutGEX'] / 10**9

c2_gex6_strike = gex_levels.index[5]
c2_gex6_call = gex_levels.iloc[5]['CallGEX'] / 10**9
c2_gex6_put = gex_levels.iloc[5]['PutGEX'] / 10**9

# CHART 3 - Gamma Profile (Gamma Flip, Max/Min, Vol Trigger)
# Gamma Flip (Intersection of Green and Red lines)
c3_gamma_flip_strike = gamma_at_spot_strike_green_red
c3_gamma_flip_value = gamma_at_spot_green_red


# Max Positive Gamma
if len(totalGammaExFri) > 0:
    max_gamma_positive_value = np.max(totalGammaExFri)
    max_gamma_positive_index = np.argmax(totalGammaExFri)
    max_gamma_positive_strike = levels[max_gamma_positive_index]
else:
    max_gamma_positive_value = 0
    max_gamma_positive_strike = 0

# Min Negative Gamma
if len(totalGammaExFri) > 0:
    min_gamma_negative_value = np.min(totalGammaExFri)
    min_gamma_negative_index = np.argmin(totalGammaExFri)
    min_gamma_negative_strike = levels[min_gamma_negative_index]
else:
    min_gamma_negative_value = 0
    min_gamma_negative_strike = 0

c3_max_pos_value = max_gamma_positive_value
c3_max_pos_strike = max_gamma_positive_strike
c3_min_neg_value = min_gamma_negative_value
c3_min_neg_strike = min_gamma_negative_strike

# Vol Trigger (zero gamma cross interpolado)
# Encontrar onde totalGammaExFri cruza zero
zeroGamma = None
vol_trigger_value_at_flip_exfri = None

for i in range(len(totalGammaExFri)-1):
    if (totalGammaExFri[i] < 0 and totalGammaExFri[i+1] > 0) or \
       (totalGammaExFri[i] > 0 and totalGammaExFri[i+1] < 0):
        # Interpolação linear para encontrar o ponto exato
        zeroGamma = np.interp(0, [totalGammaExFri[i], totalGammaExFri[i+1]],
                               [levels[i], levels[i+1]])
        vol_trigger_value_at_flip_exfri = 0
        break

c3_vol_trigger_value = vol_trigger_value_at_flip_exfri if vol_trigger_value_at_flip_exfri is not None else 0
c3_vol_trigger_strike = zeroGamma if zeroGamma is not None else 0

# Dados gerais
spot_price = spotPrice
total_gamma = df['TotalGamma'].sum()
update_date = todayDate.strftime('%d %b %Y 00:00')

# ==================== GERAÇÃO DA LINHA ÚNICA ====================
# Criar a string com TODOS os dados separados por vírgula
data_string = f"{spot_price},{total_gamma},{update_date},"
data_string += f"{c1_put_wall_value},{c1_put_wall_strike},"
data_string += f"{c1_lg1_value},{c1_lg1_strike},"
data_string += f"{c1_lg2_value},{c1_lg2_strike},"
data_string += f"{c1_call_wall_value},{c1_call_wall_strike},"
data_string += f"{c1_lg3_value},{c1_lg3_strike},"
data_string += f"{c1_lg4_value},{c1_lg4_strike},"
data_string += f"{c2_gex1_strike},{c2_gex1_call},{c2_gex1_put},"
data_string += f"{c2_gex2_strike},{c2_gex2_call},{c2_gex2_put},"
data_string += f"{c2_gex3_strike},{c2_gex3_call},{c2_gex3_put},"
data_string += f"{c2_gex4_strike},{c2_gex4_call},{c2_gex4_put},"
data_string += f"{c2_gex5_strike},{c2_gex5_call},{c2_gex5_put},"
data_string += f"{c2_gex6_strike},{c2_gex6_call},{c2_gex6_put},"
data_string += f"{c3_gamma_flip_value},{c3_gamma_flip_strike},"
data_string += f"{c3_max_pos_value},{c3_max_pos_strike},"
data_string += f"{c3_min_neg_value},{c3_min_neg_strike},"
data_string += f"{c3_vol_trigger_value},{c3_vol_trigger_strike}"

# ==================== EXIBIR RESULTADOS ====================
print("📊 DADOS COLETADOS:")
print(f"\n--- CHART 1: Spot Gamma Levels ---")
print(f"Put Wall: {c1_put_wall_value:.4f} at {c1_put_wall_strike:.0f}")
print(f"Large Gamma 1: {c1_lg1_value:.4f} at {c1_lg1_strike:.0f}")
print(f"Large Gamma 2: {c1_lg2_value:.4f} at {c1_lg2_strike:.0f}")
print(f"Call Wall: {c1_call_wall_value:.4f} at {c1_call_wall_strike:.0f}")
print(f"Large Gamma 3: {c1_lg3_value:.4f} at {c1_lg3_strike:.0f}")
print(f"Large Gamma 4: {c1_lg4_value:.4f} at {c1_lg4_strike:.0f}")

print(f"\n--- CHART 2: Top 6 GEX Levels ---")
for i in range(1, 7):
    strike = eval(f"c2_gex{i}_strike")
    call = eval(f"c2_gex{i}_call")
    put = eval(f"c2_gex{i}_put")
    print(f"GEX {i}: Strike {strike:.0f} | Call: {call:.4f} | Put: {put:.4f}")

print(f"\n--- CHART 3: Gamma Profile ---")
print(f"Gamma Flip: {c3_gamma_flip_value:.4f} at {c3_gamma_flip_strike:.0f}")
print(f"Max Positive Gamma: {c3_max_pos_value:.4f} at {c3_max_pos_strike:.0f}")
print(f"Min Negative Gamma: {c3_min_neg_value:.4f} at {c3_min_neg_strike:.0f}")
print(f"Vol Trigger: {c3_vol_trigger_value:.4f} at {c3_vol_trigger_strike:.0f}")

print(f"\n--- DADOS GERAIS ---")
print(f"Spot Price: {spot_price:.2f}")
print(f"Total Gamma: {total_gamma:.2f} Bn")
print(f"Update Date: {update_date}")

print("\n" + "="*80)
print("📋 LINHA ÚNICA PARA TRADINGVIEW (copie a linha abaixo):")
print("="*80)
print(data_string)
print("="*80)


🚀 GERADOR DE CÓDIGO TRADINGVIEW - GAMMA FLIP (VERSÃO SIMPLIFICADA)

📊 DADOS COLETADOS:

--- CHART 1: Spot Gamma Levels ---
Put Wall: -0.0038 at 23300
Large Gamma 1: -0.0037 at 23200
Large Gamma 2: -0.0037 at 23250
Call Wall: 0.0056 at 25900
Large Gamma 3: 0.0036 at 26400
Large Gamma 4: 0.0036 at 26100

--- CHART 2: Top 6 GEX Levels ---
GEX 1: Strike 26300 | Call: 0.0171 | Put: -0.0141
GEX 2: Strike 25900 | Call: 0.0182 | Put: -0.0126
GEX 3: Strike 26200 | Call: 0.0166 | Put: -0.0135
GEX 4: Strike 26400 | Call: 0.0168 | Put: -0.0132
GEX 5: Strike 26100 | Call: 0.0168 | Put: -0.0132
GEX 6: Strike 26000 | Call: 0.0154 | Put: -0.0136

--- CHART 3: Gamma Profile ---
Gamma Flip: 0.0000 at 26111
Max Positive Gamma: 0.0000 at 20960
Min Negative Gamma: 0.0000 at 20960
Vol Trigger: 0.0000 at 0

--- DADOS GERAIS ---
Spot Price: 26200.00
Total Gamma: -0.04 Bn
Update Date: 29 Oct 2025 00:00

📋 LINHA ÚNICA PARA TRADINGVIEW (copie a linha abaixo):
26200.0,-0.0425335308421759,29 Oct 2025 00:00,-0.003

### RESULTADOS DELTA

In [37]:
# ========================================
# GERADOR TRADINGVIEW - DELTA (ALL) - VERSÃO COMPLETA
# ========================================
print("\n" + "="*80)
print("🚀 GERADOR DE CÓDIGO TRADINGVIEW - DELTA")
print("="*80 + "\n")

# ==================== COLETA E CÁLCULO DOS DADOS - DELTA ====================

# Recalcular agregação Delta
dfAgg_delta = df.groupby(['StrikePrice']).sum(numeric_only=True)

# Recalcular Delta Exposure
dfAgg_delta['CallDEX'] = dfAgg_delta['CallDelta'] * dfAgg_delta['CallOpenInt'] * 100 * spotPrice
dfAgg_delta['PutDEX'] = dfAgg_delta['PutDelta'] * dfAgg_delta['PutOpenInt'] * 100 * spotPrice
dfAgg_delta['TotalDelta'] = (dfAgg_delta.CallDEX + dfAgg_delta.PutDEX) / 10**6  # Converting to millions

# CHART 4 - Absolute Delta Exposure (6 strikes: 3 menores + 3 maiores)
smallest_delta = dfAgg_delta.sort_values(by='TotalDelta').head(3)
largest_delta = dfAgg_delta.sort_values(by='TotalDelta').tail(3)

c4_put_wall_value_delta = smallest_delta.iloc[0]['TotalDelta'] if not smallest_delta.empty else 0.0
c4_put_wall_strike_delta = smallest_delta.index[0] if not smallest_delta.empty else 0
c4_lg1_value_delta = smallest_delta.iloc[1]['TotalDelta'] if len(smallest_delta) > 1 else 0.0
c4_lg1_strike_delta = smallest_delta.index[1] if len(smallest_delta) > 1 else 0
c4_lg2_value_delta = smallest_delta.iloc[2]['TotalDelta'] if len(smallest_delta) > 2 else 0.0
c4_lg2_strike_delta = smallest_delta.index[2] if len(smallest_delta) > 2 else 0

c4_call_wall_value_delta = largest_delta.iloc[2]['TotalDelta'] if len(largest_delta) > 2 else 0.0
c4_call_wall_strike_delta = largest_delta.index[2] if len(largest_delta) > 2 else 0
c4_lg3_value_delta = largest_delta.iloc[1]['TotalDelta'] if len(largest_delta) > 1 else 0.0
c4_lg3_strike_delta = largest_delta.index[1] if len(largest_delta) > 1 else 0
c4_lg4_value_delta = largest_delta.iloc[0]['TotalDelta'] if not largest_delta.empty else 0.0
c4_lg4_strike_delta = largest_delta.index[0] if not largest_delta.empty else 0

# CHART 5 - DEX Levels (Top 6 strikes por DEX absoluto)
dfAgg_delta['AbsoluteTotalDEX'] = dfAgg_delta['CallDEX'].abs() + dfAgg_delta['PutDEX'].abs()
dfAgg_delta_sorted_dex = dfAgg_delta.sort_values(by='AbsoluteTotalDEX', ascending=False)
dex_levels_delta = dfAgg_delta_sorted_dex.head(6)

c5_dex1_strike_delta = dex_levels_delta.index[0] if not dex_levels_delta.empty else 0
c5_dex1_call_delta = (dex_levels_delta.iloc[0]['CallDEX'] / 10**6) if not dex_levels_delta.empty else 0.0
c5_dex1_put_delta = (dex_levels_delta.iloc[0]['PutDEX'] / 10**6) if not dex_levels_delta.empty else 0.0

c5_dex2_strike_delta = dex_levels_delta.index[1] if len(dex_levels_delta) > 1 else 0
c5_dex2_call_delta = (dex_levels_delta.iloc[1]['CallDEX'] / 10**6) if len(dex_levels_delta) > 1 else 0.0
c5_dex2_put_delta = (dex_levels_delta.iloc[1]['PutDEX'] / 10**6) if len(dex_levels_delta) > 1 else 0.0

c5_dex3_strike_delta = dex_levels_delta.index[2] if len(dex_levels_delta) > 2 else 0
c5_dex3_call_delta = (dex_levels_delta.iloc[2]['CallDEX'] / 10**6) if len(dex_levels_delta) > 2 else 0.0
c5_dex3_put_delta = (dex_levels_delta.iloc[2]['PutDEX'] / 10**6) if len(dex_levels_delta) > 2 else 0.0

c5_dex4_strike_delta = dex_levels_delta.index[3] if len(dex_levels_delta) > 3 else 0
c5_dex4_call_delta = (dex_levels_delta.iloc[3]['CallDEX'] / 10**6) if len(dex_levels_delta) > 3 else 0.0
c5_dex4_put_delta = (dex_levels_delta.iloc[3]['PutDEX'] / 10**6) if len(dex_levels_delta) > 3 else 0.0

c5_dex5_strike_delta = dex_levels_delta.index[4] if len(dex_levels_delta) > 4 else 0
c5_dex5_call_delta = (dex_levels_delta.iloc[4]['CallDEX'] / 10**6) if len(dex_levels_delta) > 4 else 0.0
c5_dex5_put_delta = (dex_levels_delta.iloc[4]['PutDEX'] / 10**6) if len(dex_levels_delta) > 4 else 0.0

c5_dex6_strike_delta = dex_levels_delta.index[5] if len(dex_levels_delta) > 5 else 0
c5_dex6_call_delta = (dex_levels_delta.iloc[5]['CallDEX'] / 10**6) if len(dex_levels_delta) > 5 else 0.0
c5_dex6_put_delta = (dex_levels_delta.iloc[5]['PutDEX'] / 10**6) if len(dex_levels_delta) > 5 else 0.0

# CHART 6 - Delta Profile (Delta Flip, Max/Min, Vol Trigger)
# Delta Flip (Ex-Next Monthly Expiry)
c6_delta_flip_value = delta_at_spot_delta_fri
c6_delta_flip_strike = delta_at_spot_strike_delta_fri

# Max Positive Delta
max_delta_positive_value = np.max(totalDeltaExFri) if len(totalDeltaExFri) > 0 else 0
max_delta_positive_index = np.argmax(totalDeltaExFri) if len(totalDeltaExFri) > 0 else 0
max_delta_positive_strike = levels_delta[max_delta_positive_index] if len(totalDeltaExFri) > 0 else 0

# Min Negative Delta
min_delta_negative_value = np.min(totalDeltaExFri) if len(totalDeltaExFri) > 0 else 0
min_delta_negative_index = np.argmin(totalDeltaExFri) if len(totalDeltaExFri) > 0 else 0
min_delta_negative_strike = levels_delta[min_delta_negative_index] if len(totalDeltaExFri) > 0 else 0

# Vol Trigger Delta (zero delta cross)
zeroDelta = None
delta_vol_trigger_value_at_flip = None

for i in range(len(totalDeltaExFri)-1):
    if (totalDeltaExFri[i] < 0 and totalDeltaExFri[i+1] > 0) or \
       (totalDeltaExFri[i] > 0 and totalDeltaExFri[i+1] < 0):
        zeroDelta = np.interp(0, [totalDeltaExFri[i], totalDeltaExFri[i+1]],
                               [levels_delta[i], levels_delta[i+1]])
        delta_vol_trigger_value_at_flip = 0
        break

c6_max_pos_value_delta = max_delta_positive_value
c6_max_pos_strike_delta = max_delta_positive_strike
c6_min_neg_value_delta = min_delta_negative_value
c6_min_neg_strike_delta = min_delta_negative_strike
c6_vol_trigger_value_delta = delta_vol_trigger_value_at_flip if delta_vol_trigger_value_at_flip is not None else 0
c6_vol_trigger_strike_delta = zeroDelta if zeroDelta is not None else 0

# Dados gerais Delta
total_delta = df['TotalDelta'].sum()

# ==================== GERAÇÃO DA LINHA ÚNICA - DELTA ====================
data_string_delta = f"{spot_price},{total_delta},{update_date},"
data_string_delta += f"{c4_put_wall_value_delta},{c4_put_wall_strike_delta},"
data_string_delta += f"{c4_lg1_value_delta},{c4_lg1_strike_delta},"
data_string_delta += f"{c4_lg2_value_delta},{c4_lg2_strike_delta},"
data_string_delta += f"{c4_call_wall_value_delta},{c4_call_wall_strike_delta},"
data_string_delta += f"{c4_lg3_value_delta},{c4_lg3_strike_delta},"
data_string_delta += f"{c4_lg4_value_delta},{c4_lg4_strike_delta},"
data_string_delta += f"{c5_dex1_strike_delta},{c5_dex1_call_delta},{c5_dex1_put_delta},"
data_string_delta += f"{c5_dex2_strike_delta},{c5_dex2_call_delta},{c5_dex2_put_delta},"
data_string_delta += f"{c5_dex3_strike_delta},{c5_dex3_call_delta},{c5_dex3_put_delta},"
data_string_delta += f"{c5_dex4_strike_delta},{c5_dex4_call_delta},{c5_dex4_put_delta},"
data_string_delta += f"{c5_dex5_strike_delta},{c5_dex5_call_delta},{c5_dex5_put_delta},"
data_string_delta += f"{c5_dex6_strike_delta},{c5_dex6_call_delta},{c5_dex6_put_delta},"
data_string_delta += f"{c6_delta_flip_value},{c6_delta_flip_strike},"
data_string_delta += f"{c6_max_pos_value_delta},{c6_max_pos_strike_delta},"
data_string_delta += f"{c6_min_neg_value_delta},{c6_min_neg_strike_delta},"
data_string_delta += f"{c6_vol_trigger_value_delta},{c6_vol_trigger_strike_delta}"

# ==================== EXIBIR RESULTADOS - DELTA ====================
print("📊 DADOS COLETADOS - DELTA:")
print(f"\n--- CHART 4: Spot Delta Levels ---")
print(f"Put Delta Wall: {c4_put_wall_value_delta:.4f} at {c4_put_wall_strike_delta:.0f}")
print(f"Call Delta Wall: {c4_call_wall_value_delta:.4f} at {c4_call_wall_strike_delta:.0f}")

print(f"\n--- CHART 5: Top 6 DEX Levels ---")
for i in range(1, 7):
    strike = eval(f"c5_dex{i}_strike_delta")
    call = eval(f"c5_dex{i}_call_delta")
    put = eval(f"c5_dex{i}_put_delta")
    print(f"DEX {i}: Strike {strike:.0f} | Call: {call:.4f} | Put: {put:.4f}")

print(f"\n--- CHART 6: Delta Profile ---")
print(f"Delta Flip: {c6_delta_flip_value:.4f} at {c6_delta_flip_strike:.0f}")
print(f"Max Positive Delta: {c6_max_pos_value_delta:.4f} at {c6_max_pos_strike_delta:.0f}")
print(f"Min Negative Delta: {c6_min_neg_value_delta:.4f} at {c6_min_neg_strike_delta:.0f}")

print(f"\n--- DADOS GERAIS - DELTA ---")
print(f"Total Delta: {total_delta:.2f} Million")

print("\n" + "="*80)
print("📋 LINHA ÚNICA PARA TRADINGVIEW - DELTA (copie a linha abaixo):")
print("="*80)
print(data_string_delta)
print("="*80)


🚀 GERADOR DE CÓDIGO TRADINGVIEW - DELTA

📊 DADOS COLETADOS - DELTA:

--- CHART 4: Spot Delta Levels ---
Put Delta Wall: -236.9529 at 35500
Call Delta Wall: 244.8308 at 22250

--- CHART 5: Top 6 DEX Levels ---
DEX 1: Strike 23300 | Call: 259.0117 | Put: -28.1349
DEX 2: Strike 23500 | Call: 256.0359 | Put: -30.1573
DEX 3: Strike 23200 | Call: 259.3663 | Put: -26.7358
DEX 4: Strike 23250 | Call: 259.1987 | Put: -26.8865
DEX 5: Strike 23600 | Call: 252.9838 | Put: -32.8439
DEX 6: Strike 23100 | Call: 259.6500 | Put: -25.3926

--- CHART 6: Delta Profile ---
Delta Flip: 0.0000 at 26111
Max Positive Delta: 0.0000 at 20960
Min Negative Delta: 0.0000 at 20960

--- DADOS GERAIS - DELTA ---
Total Delta: 34.37 Million

📋 LINHA ÚNICA PARA TRADINGVIEW - DELTA (copie a linha abaixo):
26200.0,34.37116921285791,29 Oct 2025 00:00,-236.9529282813383,35500.0,-236.44154436857065,35000.0,-235.8586922571598,34500.0,244.8307954092823,22250.0,244.3647822175696,22200.0,243.53865541643924,22300.0,23300.0,259.01